# 2.2 — La descente de gradient : comment un modèle apprend

**Navigation** : [<< 2.1-Workflow-ML](2.1-Workflow-ML.ipynb) | [2.3-Regression-lineaire-logistique >>](2.3-Regression-lineaire-logistique.ipynb) | [Index](../README.md) | [Version Python](2.2-Descente-de-gradient.ipynb)

**Kernel** : .NET C# — Python.NET appelle les vrais jeux et estimateurs scikit-learn ; définir PYTHONNET_PYDLL et les chemins CPython si nécessaire.

## Introduction

Dans le notebook **2.1**, nous avons utilisé la méthode `fit()` de scikit-learn comme une boîte noire : on lui donne des données, elle « apprend » un modèle. Mais que se passe-t-il réellement à l'intérieur de cette boîte ? De nombreux modèles utilisent une méthode itérative comme la **descente de gradient**, mais `fit()` peut aussi employer un solveur de moindres carrés. Il s'agit d'une méthode d'optimisation qui ajuste itérativement les paramètres du modèle afin de **réduire progressivement l'erreur** qu'il commet.

Dans ce notebook, nous ouvrons la boîte noire : au lieu d'appeler `fit()`, nous **implémentons nous-mêmes** la descente de gradient sur une régression linéaire. Nous verrons concrètement comment chaque paramètre est mis à jour, et pourquoi le **learning rate** (le pas de descente) est l'hyperparamètre le plus délicat à régler.

### Objectifs d'apprentissage

À la fin de ce notebook, vous saurez :
1. Définir la fonction de coût **MSE** et expliquer pourquoi on la minimise.
2. Implémenter le **pas de descente de gradient** *from scratch*.
3. Observer l'effet du **learning rate** (convergence lente / bonne / divergence).
4. Vérifier que la descente de gradient retrouve les coefficients de `LinearRegression`.

**Durée indicative** : 60 à 90 minutes.

### Prérequis

- **2.1-Workflow-ML** : métriques d'évaluation, séparation train/test, surapprentissage.
- **Dérivées partielles** : un niveau conceptuel suffit (on manipule le gradient sans calcul formel lourd).

> **Référence.** Cauchy, A.-L. (1847), *Méthode générale pour la résolution des systèmes d'équations simultanées*, Comptes Rendus de l'Académie des Sciences 25:536-538. La méthode de descente dans la direction opposée au gradient y est introduite — c'est l'ancêtre de tout l'apprentissage automatique par optimisation.


In [1]:
#r "nuget: PythonNet, 3.0.5"
using System; using System.Linq; using System.Collections.Generic; using System.Globalization; using System.Text; using System.Net; using Python.Runtime; using Microsoft.DotNet.Interactive.Formatting;
CultureInfo.CurrentCulture=CultureInfo.InvariantCulture;
if(!PythonEngine.IsInitialized){var p=Environment.GetEnvironmentVariable("PYTHONNET_PYDLL");if(string.IsNullOrWhiteSpace(p))throw new Exception("Définir PYTHONNET_PYDLL");Runtime.PythonDLL=p;PythonEngine.Initialize();PythonEngine.BeginAllowThreads();}
record PlotSvg(string Svg); Formatter.Register<PlotSvg>((p,w)=>w.Write(p.Svg),"text/html");
string F(double x)=>x.ToString("0.###",CultureInfo.InvariantCulture); string E(string s)=>WebUtility.HtmlEncode(s);
string Plot(string title,string xl,string yl,List<(string name,string color,double[] x,double[] y)> data,bool logY=false,bool points=false){
 var valid=data.SelectMany(d=>d.x.Zip(d.y,(x,y)=>(x,y:logY?Math.Log10(y):y))).Where(p=>double.IsFinite(p.x)&&double.IsFinite(p.y)).ToArray();double xmin=valid.Min(p=>p.x),xmax=valid.Max(p=>p.x),ymin=valid.Min(p=>p.y),ymax=valid.Max(p=>p.y);if(xmax==xmin)xmax=xmin+1;if(ymax==ymin)ymax=ymin+1;double px(double x)=>70+500*(x-xmin)/(xmax-xmin);double py(double y)=>380-300*((logY?Math.Log10(y):y)-ymin)/(ymax-ymin);
 var s=new StringBuilder($"<svg xmlns='http://www.w3.org/2000/svg' width='700' height='500'><rect width='700' height='500' fill='white'/><text x='350' y='24' text-anchor='middle'>{E(title)}</text><rect x='70' y='80' width='500' height='300' fill='white' stroke='black'/>");
 for(int i=0;i<=4;i++){double x=xmin+(xmax-xmin)*i/4,y=ymin+(ymax-ymin)*i/4;s.Append($"<text x='{F(px(x))}' y='398' text-anchor='middle' font-size='10'>{F(x)}</text><text x='65' y='{F(380-300.0*i/4)}' text-anchor='end' font-size='10'>{F(y)}</text>");}
 s.Append($"<text x='320' y='425' text-anchor='middle'>{E(xl)}</text><text transform='translate(18 230) rotate(-90)' text-anchor='middle'>{E(yl+(logY?" (log10)":""))}</text>");int k=0;foreach(var d in data){var pairs=d.x.Zip(d.y,(x,y)=>(x,y)).Where(p=>double.IsFinite(p.x)&&double.IsFinite(p.y)&&(!logY||p.y>0)).ToArray();if(points && k==0){foreach(var p in pairs)s.Append($"<circle cx='{F(px(p.x))}' cy='{F(py(p.y))}' r='2.5' fill='{d.color}'/>");}else{s.Append($"<polyline fill='none' stroke='{d.color}' stroke-width='2' points='{string.Join(" ",pairs.Select(p=>F(px(p.x))+","+F(py(p.y))))}'/>");}s.Append($"<text x='80' y='{F(449+k*14)}' font-size='11' fill='{d.color}'>{E(d.name)}</text>");k++;}return s+"</svg>";}
var X=new double[100];var Y=new double[100];double ws=0,bs=0;using(Py.GIL()){dynamic ds=Py.Import("sklearn.datasets"),lin=Py.Import("sklearn.linear_model"),np=Py.Import("numpy"),sk=Py.Import("sklearn"),sys=Py.Import("sys");using PyObject pair=ds.make_regression(n_samples:100,n_features:1,noise:20.0,bias:3.0,random_state:42);using var x=pair.GetItem(0);using var y=pair.GetItem(1);using var xa=x.InvokeMethod("tolist");using var ya=y.InvokeMethod("tolist");X=xa.As<double[][]>().Select(a=>a[0]).ToArray();Y=ya.As<double[]>();using PyObject model=lin.LinearRegression();((dynamic)model).fit(x,y);ws=(double)((dynamic)model).coef_[0];bs=(double)((dynamic)model).intercept_;Console.WriteLine($".NET {Environment.Version} | Python {sys.version} | NumPy {np.__version__} | sklearn {sk.__version__} | Python.NET {typeof(PythonEngine).Assembly.GetName().Version}");}


Installed Packages PythonNet, 3.0.5

.NET 10.0.14 | Python 3.13.16 (main, Oct 14 2026, 01:04:01) [Clang 23.1.3 ] | NumPy 2.4.4 | sklearn 1.9.1 | Python.NET 3.0.5.0


### Auto-évaluation

Ce carnet porte un dispositif d'auto-évaluation **formatif** : quatre questions,
placées avant le parcours (diagnostic de prérequis), au milieu (vérification de
la notion) et à la fin (transfert). Aucune note n'est stockée, aucune réponse
n'est enregistrée.

Chaque cellule de question s'exécute sans erreur tant que vous n'avez pas
répondu ; la correction s'affiche **uniquement** lorsque vous remplacez
`response: null` par la lettre de votre choix, puis ré-exécutez la cellule.


In [2]:
void Question(string q,string[] choices,string answer,string explanation,string response=null){Console.WriteLine(q);for(int i=0;i<choices.Length;i++)Console.WriteLine($"{(char)('A'+i)}. {choices[i]}");if(response==null)Console.WriteLine("Choisir une lettre puis relancer pour voir la correction.");else Console.WriteLine((response.ToUpperInvariant()==answer?"Correct. ":"À revoir. ")+explanation);}
Console.WriteLine("Auto-évaluation formative : aucune réponse n'est enregistrée.");


Auto-évaluation formative : aucune réponse n'est enregistrée.


### Avant de commencer — question de diagnostic


In [3]:
Question("Que fait la descente de gradient ?",new[]{"Elle ajuste les coefficients en descendant un coût, pas à pas","Elle résout une équation exacte en une étape","Elle trie les données"},"A","C'est une méthode itérative.", response: null); // Remplacer null par une lettre.


Que fait la descente de gradient ?


A. Elle ajuste les coefficients en descendant un coût, pas à pas


B. Elle résout une équation exacte en une étape


C. Elle trie les données


Choisir une lettre puis relancer pour voir la correction.


## 1. Le problème : la régression linéaire comme un problème de minimisation

Une régression linéaire prédit une cible `y` à partir d'une caractéristique `x` par la relation :

$$\hat{y} = w \cdot x + b$$

où `w` (le poids) et `b` (le biais) sont les **paramètres** que le modèle doit « apprendre ». Apprendre, ici, signifie **trouver les valeurs de `w` et `b` qui rendent l'erreur moyenne aussi petite que possible**. Cette erreur se mesure par la **Mean Squared Error** (MSE, erreur quadratique moyenne) :

$$\mathrm{MSE}(w, b) = \frac{1}{n} \sum_{i=1}^{n} \big( \hat{y}_i - y_i \big)^2 = \frac{1}{n} \sum_{i=1}^{n} \big( w \cdot x_i + b - y_i \big)^2$$

Apprendre le modèle revient donc à **minimiser** la fonction de coût MSE. Pour une régression linéaire, cette fonction est **convexe** (en forme de bol) : elle possède un minimum global ; celui-ci est unique si la matrice de régression augmentée est de rang plein. Un pas adapté permet à la descente de converger ici.

> **Référence.** Gauss (1809), *Theoria Motus Corporum Coelestium* — la méthode des moindres carrés (MCO/MSE) est la fondation historique de la régression linéaire ; voir aussi Hastie, Tibshirani & Friedman (2009), *The Elements of Statistical Learning*, Springer (2e éd.), section 3.2.


In [4]:
display(new PlotSvg(Plot("Données de régression","x","y",new(){("données","steelblue",X,Y)},points:true)));


Données de régression -2.62 -123.704 -1.502 -62.934 -0.384 -2.163 0.734 58.607 1.852 119.377 x y données

## 2. La fonction de coût (MSE) et son gradient

Pour minimiser la MSE, la descente de gradient a besoin de savoir **dans quelle direction l'erreur augmente**, afin de partir dans la direction opposée. C'est exactement le rôle du **gradient** : le vecteur des dérivées partielles de la fonction de coût par rapport à chaque paramètre. Pour la MSE on obtient, après dérivation :

$$\frac{\partial \mathrm{MSE}}{\partial w} = -\frac{2}{n} \sum_{i=1}^{n} x_i \big(y_i - (w \cdot x_i + b)\big) \qquad \frac{\partial \mathrm{MSE}}{\partial b} = -\frac{2}{n} \sum_{i=1}^{n} \big(y_i - (w \cdot x_i + b)\big)$$

La règle de mise à jour de la descente de gradient est alors :

$$w \leftarrow w - \mathrm{lr} \cdot \frac{\partial \mathrm{MSE}}{\partial w} \qquad b \leftarrow b - \mathrm{lr} \cdot \frac{\partial \mathrm{MSE}}{\partial b}$$

On descend dans la **direction opposée au gradient**, d'un pas contrôlé par le **learning rate** `lr`. Répétée un grand nombre de fois, cette mise à jour fait glisser `(w, b)` vers le fond du bol = le minimum de l'erreur.

> **Référence.** Nocedal, J. & Wright, S.J. (2006), *Numerical Optimization*, Springer (2e éd.), chapitres 2-3. La convergence, le choix du pas (learning rate) et la condition de Lipschitz de la fonction de coût y sont analysés.


In [5]:
double Mse(double[] a,double[] b)=>a.Zip(b,(u,v)=>(u-v)*(u-v)).Average();
(double dw,double db) Grad(double[] x,double[] y,double w,double b){double dw=0,db=0;for(int i=0;i<y.Length;i++){double e=w*x[i]+b-y[i];dw+=e*x[i];db+=e;}return (2*dw/y.Length,2*db/y.Length);}
var g0=Grad(X,Y,0,0);Console.WriteLine($"Au départ MSE={Mse(Y,new double[Y.Length]):F4}, dw={g0.dw:F4}, db={g0.db:F4}");


Au départ MSE=2126.3453, dw=-76.8802, db=-0.8712


### Exercice 1 : calculer le coût MSE

Complétez MseEtudiant pour renvoyer la moyenne des carrés des écarts. Les tableaux vides, de longueurs différentes ou contenant des valeurs non finies doivent être rejetés par null ou une erreur de validation.

- Indice : parcourez les deux tableaux, accumulez les carrés puis divisez par le nombre d'éléments.
- Étape 1 : compléter MseEtudiant.
- Étape 2 : la cellule appelle votre fonction au point initial et vérifie plusieurs exemples.


In [6]:
double? MseEtudiant(double[] a,double[] b){return null; /* TODO : moyenne des carrés */}
bool VerifierMse(Func<double[],double[],double?> f){var tests=new[]{(new[]{1.0,-2,3},new[]{0.0,2,1},7.0),(new[]{-3.0,2},new[]{1.0,-1},12.5)};if(!tests.All(t=>{var v=f(t.Item1,t.Item2);return v.HasValue&&double.IsFinite(v.Value)&&Math.Abs(v.Value-t.Item3)<1e-10;}))return false;
 foreach(var t in new[]{(Array.Empty<double>(),Array.Empty<double>()),(new[]{1.0,double.NaN},new[]{0.0,1.0}),(new[]{1.0,2.0},new[]{1.0})}){try{if(f(t.Item1,t.Item2)!=null)return false;}catch(ArgumentException){}}return true;}
var coutTest=MseEtudiant(Y,new double[Y.Length]);Console.WriteLine(coutTest is null?"Exercice 1 à compléter":$"Fixtures correctes : {VerifierMse(MseEtudiant)} ; MSE initiale correcte : {double.IsFinite(coutTest.Value)&&Math.Abs(coutTest.Value-Mse(Y,new double[Y.Length]))<1e-8}");


Exercice 1 à compléter


## 3. La boucle de descente de gradient

Nous avons la fonction de coût et son gradient. Il ne reste qu'à **boucler** : à chaque itération, on calcule le gradient à la position courante, puis on met à jour les paramètres en faisant un petit pas dans la direction opposée. Répétée suffisamment de fois, cette boucle **est** l'apprentissage.

Le **learning rate** `lr` fixe la taille de ce pas :

- **Trop petit** : la descente est **lente**, il faut beaucoup d'itérations pour atteindre le minimum.
- **Bon** : la descente converge **rapidement et sûrement** vers le fond du bol.
- **Trop grand** : les pas sont exagérés, la MSE **oscille, puis diverge** (le coût explose).


In [7]:
(List<double> w,List<double> b,List<double> loss) GD(double[] x,double[] y,double lr,int n){double w=0,b=0;var wh=new List<double>{w};var bh=new List<double>{b};var losses=new List<double>{Mse(y,x.Select(z=>w*z+b).ToArray())};for(int i=0;i<n;i++){var g=Grad(x,y,w,b);w-=lr*g.dw;b-=lr*g.db;wh.Add(w);bh.Add(b);losses.Add(Mse(y,x.Select(z=>w*z+b).ToArray()));}return(wh,bh,losses);}
var hist=GD(X,Y,.05,100);double wf=hist.w.Last(),bf=hist.b.Last();Console.WriteLine($"GD final : w={wf:F4}, b={bf:F4}, MSE={hist.loss.Last():F4}");


GD final : w=47.1209, b=5.3247, MSE=312.2171


## 4. Visualiser la descente : la courbe de coût

Le moyen le plus immédiat de **voir** l'apprentissage est de tracer la MSE en fonction du numéro d'itération. Si tout se passe bien, cette courbe **décroît de manière monotone** vers un plateau : c'est la convergence rendue visible. Sur le même graphique, on superpose la droite ajustée aux données pour constater que le modèle a bien « appris » la relation linéaire.


In [8]:
var its=Enumerable.Range(0,hist.loss.Count).Select(i=>(double)i).ToArray();display(new PlotSvg(Plot("Convergence de la descente","itération","MSE",new(){("GD","steelblue",its,hist.loss.ToArray())},true)));var lineX=new[]{X.Min(),X.Max()};display(new PlotSvg(Plot("Droite ajustée","x","y",new(){("données","steelblue",X,Y),("droite GD","red",lineX,lineX.Select(x=>wf*x+bf).ToArray())},points:true)));


Convergence de la descente 0 2.494 25 2.703 50 2.911 75 3.119 100 3.328 itération MSE (log10) <polyline fill='none' stroke='steelblue' stroke-width='2' points='70,80 75,102.366 80,124.047 85,144.979 90,165.097 95,184.335 100,202.633 105,219.937 110,236.202 115,251.391 120,265.484 125,278.471 130,290.36 135,301.171 140,310.937 145,319.706 150,327.532 155,334.477 160,340.61 165,345.998 170,350.713 175,354.822 180,358.391 185,361.482 190,364.15 195,366.449 200,368.426 205,370.122 210,371.575 215,372.819 220,373.882 225,374.79 230,375.565 235,376.226 240,376.789 245,377.268 250,377.677 255,378.024 260,378.32 265,378.571 270,378.785 275,378.967 280,379.122 285,379.254 290,379.366 295,379.461 300,379.542 305,379.61 310,379.669 315,379.719 320,379.761 325,379.797 330,379.827 335,379.853 340,379.875 345,379.894 350,379.91 355,379.923 360,379.935 365,379.945 370,379.953 375,379.96 380,379.966 385,379.971 390,379.975 395,379.979 400,379.982 405,379.985 410,379.987 415,379.989 420,379.991 425,379.992 430,379.993 435,379.994 440,379.995 445,379.996 450,379.997 455,379.997 460,379.998 465,379.998 470,379.998 475,379.999 480,379.999 485,379.999 490,379.999 495,379.999 500,379.999 505,379.999 510,380 515,380 520,380 525,380 530,380 535,380 540,380 545,380 550,380 555,380 560,380 565,380 570,380'/> GD

Droite ajustée -2.62 -123.704 -1.502 -62.934 -0.384 -2.163 0.734 58.607 1.852 119.377 x y données droite GD

### Lecture des deux figures : une descente rapide, puis un plateau

La première figure (échelle **log** sur l'axe vertical) raconte les deux phases typiques d'une descente de gradient : une chute brutale des premières itérations — chaque pas quitte une zone de forte pente et divise le coût par un facteur visible même en log — puis un long plateau où le gradient, devenu petit, n'offre plus qu'une pente faible à suivre. C'est un compromis explicite : **les dernières décimales coûtent presque autant d'itérations que les premières magnitude**. La seconde figure montre l'aboutissement : la droite `y = 47,12·x + 5,32` traverse le nuage — deux paramètres seulement, ajustés itération après itération sans jamais résoudre d'équation.

### Vérification 1 — ce que dit le gradient


In [9]:
Question("En un point, que désigne le gradient ?",new[]{"La direction de baisse à prendre telle quelle","La direction de plus forte hausse","Le minimum global"},"B","On se déplace dans le sens opposé au gradient.", response: null); // Remplacer null par une lettre.


En un point, que désigne le gradient ?


A. La direction de baisse à prendre telle quelle


B. La direction de plus forte hausse


C. Le minimum global


Choisir une lettre puis relancer pour voir la correction.


## 5. L'effet du learning rate

Le learning rate est **le** paramètre le plus sensible de la descente de gradient. Visualisons les trois régimes côte à côte sur les mêmes données : un pas minuscule (convergence paresseuse), un bon pas (convergence franche), et un pas trop grand (divergence). La différence est frappante et illustre pourquoi régler ce paramètre est crucial en pratique.


In [10]:
var ratePlots=new List<(string name,string color,double[] x,double[] y)>();int ri=0;foreach(double lr in new[]{.001,.05,1.5}){var h=GD(X,Y,lr,100);ratePlots.Add(($"lr={lr}",new[]{"steelblue","green","red"}[ri++],its,h.loss.ToArray()));Console.WriteLine($"lr={lr} : MSE finale {h.loss.Last():G6}");}display(new PlotSvg(Plot("Effet du learning rate","itération","MSE",ratePlots,true)));


lr=0.001 : MSE finale 1622.37


lr=0.05 : MSE finale 312.217


lr=1.5 : MSE finale 5.18471E+68


Effet du learning rate 0 2.494 25 19.05 50 35.605 75 52.16 100 68.715 itération MSE (log10) <polyline fill='none' stroke='steelblue' stroke-width='2' points='70,376.225 75,376.231 80,376.236 85,376.242 90,376.247 95,376.253 100,376.258 105,376.264 110,376.269 115,376.275 120,376.28 125,376.285 130,376.291 135,376.296 140,376.302 145,376.307 150,376.313 155,376.318 160,376.323 165,376.329 170,376.334 175,376.34 180,376.345 185,376.351 190,376.356 195,376.361 200,376.367 205,376.372 210,376.378 215,376.383 220,376.388 225,376.394 230,376.399 235,376.404 240,376.41 245,376.415 250,376.421 255,376.426 260,376.431 265,376.437 270,376.442 275,376.447 280,376.453 285,376.458 290,376.463 295,376.469 300,376.474 305,376.479 310,376.485 315,376.49 320,376.495 325,376.501 330,376.506 335,376.511 340,376.517 345,376.522 350,376.527 355,376.533 360,376.538 365,376.543 370,376.549 375,376.554 380,376.559 385,376.564 390,376.57 395,376.575 400,376.58 405,376.586 410,376.591 415,376.596 420,376.601 425,376.607 430,376.612 435,376.617 440,376.622 445,376.628 450,376.633 455,376.638 460,376.643 465,376.649 470,376.654 475,376.659 480,376.664 485,376.669 490,376.675 495,376.68 500,376.685 505,376.69 510,376.695 515,376.701 520,376.706 525,376.711 530,376.716 535,376.721 540,376.727 545,376.732 550,376.737 555,376.742 560,376.747 565,376.753 570,376.758'/> lr=0.001 <polyline fill='none' stroke='green' stroke-width='2' points='70,376.225 75,376.507 80,376.78 85,377.043 90,377.296 95,377.538 100,377.768 105,377.986 110,378.191 115,378.382 120,378.559 125,378.723 130,378.872 135,379.008 140,379.131 145,379.241 150,379.34 155,379.427 160,379.504 165,379.572 170,379.632 175,379.683 180,379.728 185,379.767 190,379.801 195,379.83 200,379.854 205,379.876 210,379.894 215,379.91 220,379.923 225,379.934 230,379.944 235,379.953 240,379.96 245,379.966 250,379.971 255,379.975 260,379.979 265,379.982 270,379.985 275,379.987 280,379.989 285,379.991 290,379.992 295,379.993 300,379.994 305,379.995 310,379.996 315,379.996 320,379.997 325,379.997 330,379.998 335,379.998 340,379.998 345,379.999 350,379.999 355,379.999 360,379.999 365,379.999 370,379.999 375,379.999 380,380 385,380 390,380 395,380 400,380 405,380 410,380 415,380 420,380 425,380 430,380 435,380 440,380 445,380 450,380 455,380 460,380 465,380 470,380 475,380 480,380 485,380 490,380 495,380 500,380 505,380 510,380 515,380 520,380 525,380 530,380 535,380 540,380 545,380 550,380 555,380 560,380 565,380 570,380'/> lr=0.05 <polyline fill='none' stroke='red' stroke-width='2' points='70,376.225 75,374.871 80,373.093 85,370.837 90,368.208 95,365.367 100,362.429 105,359.45 110,356.455 115,353.454 120,350.45 125,347.446 130,344.441 135,341.436 140,338.431 145,335.426 150,332.421 155,329.416 160,326.411 165,323.406 170,320.401 175,317.396 180,314.391 185,311.386 190,308.381 195,305.376 200,302.371 205,299.366 210,296.361 215,293.356 220,290.351 225,287.346 230,284.341 235,281.336 240,278.331 245,275.326 250,272.321 255,269.316 260,266.311 265,263.306 270,260.301 275,257.296 280,254.291 285,251.286 290,248.281 295,245.276 300,242.271 305,239.266 310,236.261 315,233.256 320,230.251 325,227.246 330,224.241 335,221.236 340,218.231 345,215.226 350,212.221 355,209.216 360,206.211 365,203.206 370,200.201 375,197.196 380,194.191 385,191.186 390,188.181 395,185.176 400,182.171 405,179.166 410,176.161 415,173.156 420,170.15 425,167.145 430,164.14 435,161.135 440,158.13 445,155.125 450,152.12 455,149.115 460,146.11 465,143.105 470,140.1 475,137.095 480,134.09 485,131.085 490,128.08 495,125.075 500,122.07 505,119.065 510,116.06 515,113.055 520,110.05 525,107.045 530,104.04 535,101.035 540,98.03 545,95.025 550,92.02 555,89.015 560,86.01 565,83.005 570,80'/> lr=1.5

### Lecture : trois destins d'un même algorithme

Les trois courbes incarnent les trois régimes d'un même mécanisme. Avec `lr = 0,001`, la descente **rampe** : après 100 itérations la MSE passe d'environ 2126 à 1622, encore loin du minimum. Avec `lr = 0,05`, le coût plonge immédiatement vers le minimum — le régime utile. Avec `lr = 1,5`, la courbe **explose** : chaque pas dépasse le minimum de si loin qu'il repart de plus haut — l'erreur s'amplifie à chaque itération au lieu de se réduire, signature d'une divergence géométrique.

Pour ce jeu de données, le gradient en `w` a une pente de l'ordre de `2·x² ≈ 2`, ce qui suggère que tout `lr` au-delà de `~1,0` fait overshooter. La leçon générale : **le learning rate admissible dépend de la courbure du coût** (des données), pas d'une règle universelle — d'où les optimiseurs adaptatifs (Adam, notebook **3.x**) qui ajustent le pas paramètre par paramètre, et les *schedules* qui commencent grand puis ralentissent.

### Exercice 2 : trouver le learning rate qui fait diverger la descente

Choisissez un taux divergent, puis complétez DivergenceEtudiant : le résultat doit être vrai si la dernière perte est non finie ou supérieure à dix fois la perte initiale.

- Indice : essayez les taux 1.5, 2.0 ou 5.0 ; double.IsFinite teste la finitude.
- Étape 1 : fixer lrTropGrand.
- Étape 2 : GD construit l'historique des pertes.
- Étape 3 : le détecteur lit l'historique et les fixtures vérifient aussi des cas convergents.

Le taux choisi doit être fini et strictement positif. La validation exige aussi que l’historique réellement calculé avec ce taux soit divergent : un détecteur correct associé à un taux convergent ne termine pas l’exercice.


In [11]:
double? lrTropGrand=null; // TODO : choisir un taux fini, positif et divergent
bool? DivergenceEtudiant(double[] pertes){return null; /* TODO : dernière perte non finie ou > 10 fois la première */}
bool TauxValide(double? lr)=>lr.HasValue&&double.IsFinite(lr.Value)&&lr.Value>0;
bool VerifierDivergence(Func<double[],bool?> f){var tests=new[]{(new[]{10.0,4,2},false),(new[]{1.0,11},true),(new[]{3.0,double.NaN},true),(new[]{3.0,double.PositiveInfinity},true),(new[]{2.0,20},false),(new[]{2.0,19},false)};return tests.All(t=>f(t.Item1)==t.Item2);}
bool VerifierExerciceDivergence(Func<double[],bool?> f,double? lr){
    if(!TauxValide(lr)||!VerifierDivergence(f))return false;
    var pertes=GD(X,Y,lr.Value,100).loss.ToArray();
    bool attendu=!double.IsFinite(pertes.Last())||pertes.Last()>10*pertes[0];
    return attendu&&f(pertes)==true;
}
double[] pertesDiv=null;
if(lrTropGrand is null)Console.WriteLine("Exercice 2 à compléter");
else if(!TauxValide(lrTropGrand))Console.WriteLine("Exercice 2 : taux refusé ; choisir un nombre fini strictement positif.");
else{
    pertesDiv=GD(X,Y,lrTropGrand.Value,100).loss.ToArray();
    var diagnostic=DivergenceEtudiant(pertesDiv);
    Console.WriteLine(diagnostic is null?"Exercice 2 à compléter":$"Fixtures correctes : {VerifierDivergence(DivergenceEtudiant)} ; diagnostic {diagnostic} ; exercice validé : {VerifierExerciceDivergence(DivergenceEtudiant,lrTropGrand)}");
}


Exercice 2 à compléter


## 6. La descente de gradient retrouve-t-elle les coefficients sklearn ?

Comment savoir si notre implémentation « maison » est correcte ? La validation la plus simple consiste à comparer notre résultat à celui de scikit-learn. `LinearRegression` ne fait pas de descente de gradient : elle résout numériquement les moindres carrés denses avec `scipy.linalg.lstsq`. Ici, le rang plein et un pas stable donnent une solution unique vers laquelle la descente converge. C'est notre test de cohérence.


In [12]:
Console.WriteLine($"sklearn : w={ws:F4}, b={bs:F4}\nGD : w={wf:F4}, b={bf:F4}\nÉcarts : |dw|={Math.Abs(ws-wf):F4}, |db|={Math.Abs(bs-bf):F4}");


sklearn : w=47.1332, b=5.3302
GD : w=47.1209, b=5.3247
Écarts : |dw|=0.0124, |db|=0.0055


### Lecture : deux chemins, un même minimum

`LinearRegression` utilise un solveur numérique de moindres carrés, non une inversion explicite. Les écarts observés sont `|Δw| = 0,0124` et `|Δb| = 0,0055` après 100 pas. Ici le rang plein donne un optimum unique et le pas choisi est stable ; convexité seule ne garantit pas l'unicité. La descente approche ce minimum sans calcul exact en arithmétique finie.

Les méthodes itératives restent utiles pour de nombreux modèles, notamment lorsque les calculs directs deviennent coûteux ou que le modèle n'a pas de solution fermée.


### Vérification 2 — quand le pas est trop grand


In [13]:
Question("Avec un learning rate trop grand, qu'observe-t-on ?",new[]{"Une convergence plus lente mais garantie","Un coût qui remonte, voire diverge","Aucun effet"},"B","Un pas trop grand peut amplifier la perte.", response: null); // Remplacer null par une lettre.


Avec un learning rate trop grand, qu'observe-t-on ?


A. Une convergence plus lente mais garantie


B. Un coût qui remonte, voire diverge


C. Aucun effet


Choisir une lettre puis relancer pour voir la correction.


## 7. Pourquoi standardiser avant la descente de gradient

Lorsque les caractéristiques ont des échelles très différentes (par exemple l'une en mètres et l'autre en millimètres), la fonction de coût n'est plus un bol « rond » mais un bol **allongé**. Le gradient est dominé par la direction de grande échelle : un pas stable pour elle est souvent trop petit pour les autres, ce qui ralentit fortement la convergence.

La **standardisation** (vue avec `StandardScaler` dans le pipeline du notebook 2.1) ramène toutes les caractéristiques à une variance comparable : le conditionnement peut s’améliorer, sans garantir un bol rond. Ici on compare aussi deux taux d’apprentissage distincts. C'est l'une des raisons pour lesquelles la standardisation est quasi systématique avant l'entraînement.


In [14]:
double[][] X2=null,XS=null;double[] Y2=null;double optimum=0;
using(Py.GIL()){dynamic ds=Py.Import("sklearn.datasets"),lin=Py.Import("sklearn.linear_model"),pre=Py.Import("sklearn.preprocessing"),np=Py.Import("numpy");
 using PyObject pair=ds.make_regression(n_samples:100,n_features:2,noise:20.0,bias:3.0,random_state:42);using var x=pair.GetItem(0);using var y=pair.GetItem(1);
 using var factors=new PyList();using var v1=new PyFloat(100);using var v2=new PyFloat(1);factors.Append(v1);factors.Append(v2);
 using PyObject raw=np.multiply(x,factors);using PyObject scaled=pre.StandardScaler().fit_transform(raw);using var rawList=raw.InvokeMethod("tolist");using var scaledList=scaled.InvokeMethod("tolist");using var yList=y.InvokeMethod("tolist");X2=rawList.As<double[][]>();XS=scaledList.As<double[][]>();Y2=yList.As<double[]>();
 using PyObject model=lin.LinearRegression();((dynamic)model).fit(raw,y);using PyObject pred=((dynamic)model).predict(raw);optimum=(double)np.mean(np.square(np.subtract(y,pred)));
}
List<double> GD2(double[][] x,double[] y,double lr,int n,double stop=-1){double w0=0,w1=0,b=0;var losses=new List<double>();for(int iter=0;iter<n;iter++){double g0=0,g1=0,gb=0;for(int i=0;i<y.Length;i++){double e=w0*x[i][0]+w1*x[i][1]+b-y[i];g0+=e*x[i][0];g1+=e*x[i][1];gb+=e;}w0-=lr*2*g0/y.Length;w1-=lr*2*g1/y.Length;b-=lr*2*gb/y.Length;double loss=0;for(int i=0;i<y.Length;i++){double e=w0*x[i][0]+w1*x[i][1]+b-y[i];loss+=e*e;}loss/=y.Length;losses.Add(loss);if(stop>=0&&loss<stop)break;}return losses;}
var rawLoss=GD2(X2,Y2,.0001,200);var scaledLoss=GD2(XS,Y2,.05,200);var its2=Enumerable.Range(0,200).Select(i=>(double)i).ToArray();Console.WriteLine($"200 pas : brute={rawLoss.Last():F4}, standardisée={scaledLoss.Last():F4}");display(new PlotSvg(Plot("Standardisation et convergence","itération","MSE",new(){("brut lr=0.0001","red",its2,rawLoss.ToArray()),("standardisé lr=0.05","steelblue",its2,scaledLoss.ToArray())},true)));


200 pas : brute=5507.5068, standardisée=460.0550


Standardisation et convergence 0 2.663 49.75 2.989 99.5 3.315 149.25 3.641 199 3.967 itération MSE (log10) <polyline fill='none' stroke='red' stroke-width='2' points='70,105.463 72.513,119.964 75.025,123.633 77.538,124.52 80.05,124.753 82.563,124.834 85.075,124.881 87.588,124.92 90.101,124.957 92.613,124.994 95.126,125.031 97.638,125.067 100.151,125.104 102.663,125.141 105.176,125.177 107.688,125.214 110.201,125.25 112.714,125.287 115.226,125.324 117.739,125.36 120.251,125.397 122.764,125.433 125.276,125.47 127.789,125.507 130.302,125.543 132.814,125.58 135.327,125.616 137.839,125.653 140.352,125.689 142.864,125.726 145.377,125.763 147.889,125.799 150.402,125.836 152.915,125.872 155.427,125.909 157.94,125.945 160.452,125.982 162.965,126.019 165.477,126.055 167.99,126.092 170.503,126.128 173.015,126.165 175.528,126.201 178.04,126.238 180.553,126.275 183.065,126.311 185.578,126.348 188.09,126.384 190.603,126.421 193.116,126.457 195.628,126.494 198.141,126.53 200.653,126.567 203.166,126.604 205.678,126.64 208.191,126.677 210.704,126.713 213.216,126.75 215.729,126.786 218.241,126.823 220.754,126.859 223.266,126.896 225.779,126.932 228.291,126.969 230.804,127.006 233.317,127.042 235.829,127.079 238.342,127.115 240.854,127.152 243.367,127.188 245.879,127.225 248.392,127.261 250.905,127.298 253.417,127.334 255.93,127.371 258.442,127.407 260.955,127.444 263.467,127.48 265.98,127.517 268.492,127.554 271.005,127.59 273.518,127.627 276.03,127.663 278.543,127.7 281.055,127.736 283.568,127.773 286.08,127.809 288.593,127.846 291.106,127.882 293.618,127.919 296.131,127.955 298.643,127.992 301.156,128.028 303.668,128.065 306.181,128.101 308.693,128.138 311.206,128.174 313.719,128.211 316.231,128.247 318.744,128.284 321.256,128.32 323.769,128.357 326.281,128.393 328.794,128.43 331.307,128.466 333.819,128.503 336.332,128.539 338.844,128.576 341.357,128.612 343.869,128.649 346.382,128.685 348.894,128.722 351.407,128.758 353.92,128.795 356.432,128.831 358.945,128.867 361.457,128.904 363.97,128.94 366.482,128.977 368.995,129.013 371.508,129.05 374.02,129.086 376.533,129.123 379.045,129.159 381.558,129.196 384.07,129.232 386.583,129.269 389.095,129.305 391.608,129.342 394.121,129.378 396.633,129.415 399.146,129.451 401.658,129.487 404.171,129.524 406.683,129.56 409.196,129.597 411.709,129.633 414.221,129.67 416.734,129.706 419.246,129.743 421.759,129.779 424.271,129.816 426.784,129.852 429.296,129.888 431.809,129.925 434.322,129.961 436.834,129.998 439.347,130.034 441.859,130.071 444.372,130.107 446.884,130.143 449.397,130.18 451.91,130.216 454.422,130.253 456.935,130.289 459.447,130.326 461.96,130.362 464.472,130.399 466.985,130.435 469.497,130.471 472.01,130.508 474.523,130.544 477.035,130.581 479.548,130.617 482.06,130.653 484.573,130.69 487.085,130.726 489.598,130.763 492.111,130.799 494.623,130.836 497.136,130.872 499.648,130.908 502.161,130.945 504.673,130.981 507.186,131.018 509.698,131.054 512.211,131.09 514.724,131.127 517.236,131.163 519.749,131.2 522.261,131.236 524.774,131.272 527.286,131.309 529.799,131.345 532.312,131.382 534.824,131.418 537.337,131.454 539.849,131.491 542.362,131.527 544.874,131.564 547.387,131.6 549.899,131.636 552.412,131.673 554.925,131.709 557.437,131.745 559.95,131.782 562.462,131.818 564.975,131.855 567.487,131.891 570,131.927'/> brut lr=0.0001 <polyline fill='none' stroke='steelblue' stroke-width='2' points='70,80 72.513,100.573 75.025,120.874 77.538,140.846 80.05,160.423 82.563,179.53 85.075,198.085 87.588,215.995 90.101,233.164 92.613,249.493 95.126,264.885 97.638,279.253 100.151,292.523 102.663,304.641 105.176,315.58 107.688,325.338 110.201,333.942 112.714,341.442 115.226,347.91 117.739,353.434 120.251,358.109 122.764,362.035 125.276,365.307 127.789,368.019 130.302,370.254 132.814,372.089 135.327,373.59 137.839,374.813 140.352,375.807 142.864,376.615 145.377,377.268 147.889,377.797 150.402,378.225 152.915,378.57 155.427,378.848 157.94,379.073 160.452,379.254 162.965,379.399 165.477,379.517 167.99,379.611 17

### Lecture de la figure : deux échelles et deux taux

La première caractéristique est multipliée par 100 : son gradient contraint le pas brut à `1e-4`, tandis que la version standardisée utilise `0,05`. Les deux courbes comparent donc simultanément la transformation et le taux choisi. La standardisation améliore ici le conditionnement, sans garantir que toutes les directions deviennent parfaitement équivalentes. L'exemple guidé mesure le nombre de pas jusqu'à un même seuil.


### Exemple guidé : accélérer la convergence par la standardisation (crédité #22593)

Comptons combien d'itérations il faut à la descente de gradient pour descendre sous un seuil de perte fixé, **sans** puis **avec** standardisation, et rapportons le rapport entre les deux.

Cette correction rendue en PR #22593 a un intérêt supplémentaire : elle part du seuil `50` de l'énoncé d'origine, constate qu'il est **inatteignable** — le bruit des données (`noise=20.0`) impose une MSE minimale d'environ `460`, même à la solution numérique des moindres carrés — puis construit le seuil atteignable qui rend la comparaison mesurable (optimum + 5 %). Avant de régler un seuil de perte, on le compare toujours à la perte optimale : c'est la première leçon de cet exemple.

In [15]:
int? FirstUnder(List<double> a,double threshold){for(int i=0;i<a.Count;i++)if(a[i]<threshold)return i+1;return null;}
double threshold=50, reachable=1.05*optimum;var rs=FirstUnder(rawLoss,threshold);var ss=FirstUnder(scaledLoss,threshold);var scaledAt=FirstUnder(scaledLoss,reachable);var rawAt=FirstUnder(rawLoss,reachable);var extended=GD2(X2,Y2,.0001,100000,reachable);var longAt=FirstUnder(extended,reachable);
Console.WriteLine($"Seuil {threshold} : brut={rs?.ToString()??"aucun"}, standardisé={ss?.ToString()??"aucun"}\nMSE optimale sklearn={optimum:F4}; minima 200 pas brut={rawLoss.Min():F4}, standardisé={scaledLoss.Min():F4}\nSeuil optimum + 5%={reachable:F4} : standardisé={scaledAt}, brut 200={rawAt?.ToString()??"aucun"}, brut prolongé={longAt}, rapport={((double)longAt/scaledAt):F1}");


Seuil 50 : brut=aucun, standardisé=aucun
MSE optimale sklearn=460.0550; minima 200 pas brut=5507.5068, standardisé=460.0550
Seuil optimum + 5%=483.0577 : standardisé=29, brut 200=aucun, brut prolongé=13825, rapport=476.7


**Lecture du résultat — correction créditée #22593**

- Avec le seuil demandé (`50`), les deux comptes affichent « aucun », et ce n'est pas un défaut de la descente : les données bruitées (`noise=20.0`) impose une MSE minimale d'environ `460`, même pour la solution numérique de `LinearRegression`. Aucun modèle linéaire ne peut descendre sous `50` sur ces données. Le bon réflexe, avant de régler un seuil, est donc de le comparer à la perte optimale.
- Avec un seuil atteignable (optimum + 5 %), le contraste apparaît : la descente standardisée y arrive en une trentaine d'itérations. Sans standardisation, elle reste bloquée vers `5500` au bout des 200 itérations prévues, et il lui faut plus de 13 000 itérations pour y arriver, soit près de 480 fois plus.
- La cause est celle décrite plus haut. Le `lr` doit rester minuscule (`1e-4`) pour ne pas diverger sur la caractéristique multipliée par 100, et ce même pas est alors bien trop petit pour l'autre caractéristique, qui n'avance presque plus. Après standardisation, un seul pas convient aux deux directions.


### Exercice 3 : régler le taux d'apprentissage par balayage

L'exemple guidé montre qu'après standardisation, un `lr` de `0.05` converge en une trentaine d'itérations. Mais est-ce le meilleur pas ? Balayez plusieurs taux d'apprentissage sur le problème standardisé et comparez.

- `# étape 1` : pour chaque `lr` de `lrsABalayer`, lancer `GD2(XS, Y2, lr, 200)` et garder la perte finale.
- `# étape 2` : identifier le `lr` qui atteint la plus petite perte finale.
- `# étape 3` : si un `lr` fait diverger la descente (perte `inf` ou `nan`), le noter et commenter pourquoi.


In [16]:
var lrsABalayer=new[]{.001,.01,.1,.5,2.0};
Dictionary<double,double> pertesFinales=null; // TODO : chaque taux -> perte après 200 pas
double? meilleurLr=null; // TODO : taux avec la plus petite perte finie
bool VerifierBalayage(Dictionary<double,double> pertes,double? meilleur){if(pertes==null||pertes.Count!=lrsABalayer.Length||!lrsABalayer.All(pertes.ContainsKey)||meilleur is null)return false;var r=lrsABalayer.ToDictionary(lr=>lr,lr=>GD2(XS,Y2,lr,200).Last());foreach(var lr in lrsABalayer){double a=r[lr],b=pertes[lr];if(double.IsFinite(a)){if(!double.IsFinite(b)||Math.Abs(a-b)>1e-8*Math.Max(1,Math.Abs(a)))return false;}else if(double.IsFinite(b))return false;}var finite=lrsABalayer.Where(lr=>double.IsFinite(r[lr])).ToArray();return finite.Contains(meilleur.Value)&&Math.Abs(r[meilleur.Value]-finite.Min(lr=>r[lr]))<1e-8;}
Console.WriteLine(pertesFinales is null||meilleurLr is null?"Exercice 3 à compléter":$"Balayage correct : {VerifierBalayage(pertesFinales,meilleurLr)}");


Exercice 3 à compléter


### Après le parcours — question de transfert


In [17]:
Question("Pourquoi standardiser avant la descente ?",new[]{"Pour rendre les pas toujours plus fins","Pour améliorer souvent le conditionnement de la surface","Pour supprimer les variables inutiles"},"B","Des échelles comparables peuvent permettre un pas plus grand ; cela dépend aussi des corrélations.", response: null); // Remplacer null par une lettre.


Pourquoi standardiser avant la descente ?


A. Pour rendre les pas toujours plus fins


B. Pour améliorer souvent le conditionnement de la surface


C. Pour supprimer les variables inutiles


Choisir une lettre puis relancer pour voir la correction.


## À retenir

Nous avons ouvert la boîte noire `fit()` :

- La **fonction de coût MSE** mesure l'erreur du modèle ; l'apprentissage consiste à la minimiser.
- Le **gradient** indique la direction de la hausse d'erreur ; la descente de gradient fait un pas dans la direction **opposée**.
- Le **learning rate** contrôle la taille du pas : trop petit = lent, trop grand = divergence.
- La **standardisation** peut améliorer le conditionnement ; ici, les deux taux choisis diffèrent aussi.
- Notre implémentation *from scratch* **retrouve les coefficients de `LinearRegression`**, ce qui fournit un contrôle numérique.

Ces idées (fonction de coût, gradient, learning rate) sont universelles : ce sont les mêmes qui pilotent les réseaux de neurones profonds via la rétropropagation.

| Concept | Mesure dans ce carnet |
|---|---|
| Le coût au départ | à `w=0, b=0` : `MSE = 2126.3453`, gradient `dw = -76.8802`, `db = -0.8712` — le pas ira à l'opposé du gradient |
| La boucle converge | `w = 47.1209`, `b = 5.3247`, `MSE = 312.2171` après le budget d'itérations réglé |
| Deux chemins, un même minimum | OLS numérique (`sklearn`) : `w = 47.1332`, `b = 5.3302` ; écarts absolus `0.0124` et `0.0055` |
| Le learning rate décide du destin | trop petit : la convergence traîne ; trop grand : le coût stagne, remonte, puis diverge (section 5) |
| La standardisation change le temps, pas la destination | seuil à « optimum + 5 % » (`483.1`) : `29` itérations avec standardisation (`lr = 0.05`) contre `13825` sans (`lr = 1e-4`, en prolongeant) — environ `477` fois plus |

**La leçon transférable.** Certains estimateurs utilisent un coût, un gradient et des pas ; `LinearRegression.fit()` emploie ici un solveur numérique direct. La descente ne trouve pas un minimum exact, elle s'en approche — et la forme du coût (vallée allongée ou bol rond) décide du pas praticable, donc du nombre de pas nécessaires.

**Transition.** Le carnet suivant, [2.3-Regression-lineaire-logistique](2.3-Regression-lineaire-logistique.ipynb), réutilise cette intuition d'optimisation en passant de la régression continue (prédire un nombre) à la classification binaire (prédire une étiquette 0/1).

## Vérifiez votre compréhension

**1. La descente s'arrête à `MSE final = 312.2171`, pas à 0. Pourquoi cette valeur finale n’est-elle pas nécessairement le plancher exact ?**

<details>
<summary>Réponse</summary>

Les cibles ont été générées comme `y = w·x + b + bruit` : la part du bruit est irréductible pour tout `(w, b)`. Le `312.2171` final est une approximation, légèrement au-dessus du minimum numérique estimé, et l'écart minuscule aux coefficients numériques d'OLS (`0.0124` sur `w`, `0.0055` sur `b`) confirme que la boucle est bien descendue vers le même minimum.
</details>

**2. L'écart `|dw| = 0.0124` entre la descente et `sklearn` signifie-t-il que l'implémentation est fausse ?**

<details>
<summary>Réponse</summary>

Non. OLS résout le problème numériquement ; la descente de gradient est itérative et s'approche asymptotiquement de l'optimum. Après le budget d'itérations fixé, on se trouve à `0.0124` de la solution numérique — un écart d'ordre de grandeur millième-relatif, qui valide l'implémentation au lieu de l'infirmer. C'est le même geste de vérification que dans les carnets 2.3b et 2.4b : croiser une implémentation maison contre la référence de la bibliothèque.
</details>

**3. Avec standardisation, `29` itérations suffisent ; sans, il en faut `13825`. Attention : les learning rates diffèrent aussi (`0.05` contre `1e-4`). Que change exactement la standardisation ?**

<details>
<summary>Réponse</summary>

Elle améliore ici le conditionnement de la vallée. Sans elle, la première caractéristique (multipliée par 100) produit une forte courbure dans une direction, tandis que la seconde évolue plus lentement : un pas qui convient à l'un fait dévier l'autre, et le seul `lr` praticable est minuscule (`1e-4`). Avec des variables centrées-réduites, un même pas (`0.05`) convient aux deux poids et au biais. La comparaison mêle les deux effets — échelle et learning rate — mais c'est précisément le point : le learning rate praticable est lui-même déterminé par le conditionnement de la surface.
</details>

## Pour aller plus loin

- [README de la sous-série 02-ML-Cours](README.md) — la carte des carnets et de leurs concepts-phares.
- [README de la série DataScienceWithAgents](../README.md) — le rôle de ce socle ML dans la formation.
- [2.3-Regression-lineaire-logistique](2.3-Regression-lineaire-logistique.ipynb) — la suite immédiate : de la régression continue à la classification binaire.
- [2.8-Theorie-PAC](2.8-Theorie-PAC.ipynb) — la même intuition de coût minimisé, vue du côté théorique : ce que l'optimisation peut et ne peut pas garantir.
- [Documentation scikit-learn : `StandardScaler`](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html) — la standardisation utilisée dans la section 6.

## Références

1. Cauchy, A.-L. (1847). *Méthode générale pour la résolution des systèmes d'équations simultanées*. Comptes Rendus de l'Académie des Sciences 25:536-538. — L'ancêtre de la descente de gradient.
2. Nocedal, J. & Wright, S.J. (2006). *Numerical Optimization*. Springer (2e éd.). — Convergence, choix du pas (learning rate), analyse des méthodes de descente.
3. Hastie, T., Tibshirani, R. & Friedman, J. (2009). *The Elements of Statistical Learning*. Springer (2e éd.), section 3.2. — Régression linéaire par moindres carrés et fonction de coût MSE.
4. Rumelhart, D.E., Hinton, G.E. & Williams, R.J. (1986). *Learning representations by back-propagating errors*. Nature 323:533-536. — La descente de gradient appliquée aux réseaux de neurones (rétropropagation).
5. Pedregosa, F. et al. (2011). *Scikit-learn: Machine Learning in Python*. Journal of Machine Learning Research 12:2825-2830. — `LinearRegression` et `StandardScaler` utilisés pour la validation.
6. Deisenroth, M.P., Faisal, A.A. & Ong, C.S. (2020). *Mathematics for
Machine Learning*. Cambridge UP. — ch. 5 pose le calcul vectoriel et
les gradients (chain rule, Jacobien) qu'emploie la rétropropagation ;
ch. 7 traite l'optimisation continue avec conditions d'optimalité
(KKT), convergence de la descente de gradient et rôle du pas
d'apprentissage. C'est la référence qui unifie les trois autres
citations (Cauchy 1847 pour l'invention, Nocedal & Wright pour la
théorie de convergence, Hastie ESL pour la régression MSE) sous le
même cadre mathématique.
7. Goodfellow, I., Bengio, Y. & Courville, A. (2016). *Deep
Learning*. MIT Press. — ch. 8.1 distingue l'apprentissage de
l'optimisation pure ; ch. 8.2 nomme et formalise les pathologies
mesurées ici : la vallée allongée de la section 7 est un hessien
mal conditionné (8.2.1), le plateau de la section 4 le gradient
qui s'annule à l'approche du minimum — cas convexe des régions
plates que 8.2.2 décrit en général —, la divergence au pas trop
grand l'effet des falaises (8.2.3) ; ch. 8.3 la boucle SGD et
ses variantes. Là où Deisenroth (entrée 6) pose le cadre
mathématique, Goodfellow donne le traitement propre au deep
learning.
